<CENTER><img src="../../images/ATLASOD.gif" style="width:50%"></CENTER>

# Search for the Graviton yourself!
This notebook uses ATLAS Open Data http://opendata.atlas.cern to show you the steps to search for the Graviton yourself!

ATLAS Open Data provides open access to proton-proton collision data at the LHC for educational purposes. ATLAS Open Data resources are ideal for high-school, undergraduate and postgraduate students.

Notebooks are web applications that allow you to create and share documents that can contain for example:
1. live code
2. visualisations
3. narrative text

The idea is that cuts increase the ratio of signal (Graviton) to background ($Z, t\bar{t}, t\bar{t}V, ZZ$)

This analysis loosely follows the search for a ZZ resonance by ATLAS https://link.springer.com/article/10.1140%2Fepjc%2Fs10052-018-5686-3 (mostly Figure 4)

By the end of this notebook you will be able to:
1. search for the Graviton yourself!
2. know some general principles of a particle physics search

Feynman diagram pictures are borrowed from our friends at https://www.particlezoo.net

<CENTER><img src="images/feynman_diagrams/Graviton_feynman.png" style="width:40%"></CENTER>

<a id='contents'></a>

Contents: 

[Running a Jupyter notebook](#running) <br />
[First time setup on your computer (no need on mybinder)](#setup_computer) <br />
[To setup everytime](#setup_everytime) <br />
[Lumi, fraction, file path](#fraction) <br />
[Samples](#samples) <br />
[Changing a cut](#changing_cut) <br />
[Applying a cut](#applying_cut) <br />
[Plotting](#plotting) <br />
[What can you do to explore this analysis?](#going_further) <br />

<a id='running'></a>

## Running a Jupyter notebook

To run the whole Jupyter notebook, in the top menu click Cell -> Run All.

To propagate a change you've made to a piece of code, click Cell -> Run All Below.

You can also run a single code cell, by clicking Cell -> Run Cells, or using the keyboard shortcut Shift+Enter.

<a id='setup_computer'></a>

## First time setup on your computer (no need on mybinder)
This first cell only needs to be run the first time you open this notebook on your computer. 

If you close Jupyter and re-open on the same computer, you won't need to run this first cell again.

If you open on mybinder, you don't need to run this cell.

In [ ]:
import sys
import os.path
!pip install atlasopenmagic
from atlasopenmagic import install_from_environment
install_from_environment()

[Back to contents](#contents)

<a id='setup_everytime'></a>

## To setup everytime
Cell -> Run All Below

to be done every time you re-open this notebook

We're going to be using a number of tools to help us:
* uproot: lets us read .root files typically used in particle physics into data formats used in python
* awkward: lets us store and manipulate data with a different number of entries per event, such as leptons
* vector: lets us do calculations with 4-momenta, such as invariant masses
* numpy: provides numerical calculations such as histogramming
* matplotlib: common tool for making plots, figures, images, visualisations
* atlasopenmagic: finds the ATLAS Open Data files and their metadata

In [ ]:
import uproot # for reading .root files
import awkward as ak # to represent nested data in columnar format
import vector # for 4-momentum calculations
import time # to measure time to analyse
import numpy as np # for numerical calculations such as histogramming
import matplotlib.pyplot as plt # for plotting
from matplotlib.ticker import AutoMinorLocator,LogLocator,LogFormatterSciNotation # for minor ticks
import atlasopenmagic as atom # to find the ATLAS Open Data files and their metadata

[Back to contents](#contents)

<a id='fraction'></a>

## Lumi, fraction, file path

General definitions of luminosity and fraction of data used

In [ ]:
lumi = 36.6 # fb-1 # data size of the full release

fraction = 0.09 # reduce this is you want the code to run quicker

[Back to contents](#contents)

<a id='samples'></a>

## Samples

Samples to process, given by their dataset IDs. `atlasopenmagic` turns them into the list of files to read.

We use the `exactly4lep` skim: the files have already been filtered to include exactly 4 leptons per event, so that processing is quicker.

In [ ]:
atom.set_release('2026e-13tev') # use the 2026 ATLAS Open Data release for education
skim = 'exactly4lep' # files pre-filtered to events with exactly 4 leptons

defs = {

    'data': {
        'dids' : ['data']
    },

    r'$Z,t\bar{t}$' : { # Z + ttbar
        'dids' : [700320,700321,700322, # Z->ee
                  700323,700324,700325, # Z->mumu
                  410470], # ttbar
        'color' : "#6b59d3" # purple
    },

    r'$t\bar{t}V$' : { # ttV
        'dids' : [410155,410218,410219], # ttW, ttZ(->ee), ttZ(->mumu)
        'color' : "#f0f423" # yellow
    },

    'ZZ' : { # ZZ->llll
        'dids' : [700600],
        'color' : "#ff0000" # red
    },

    'Graviton' : {
        'dids' : [546708], # mG = 600 GeV
        'color' : "#baff8d" # green
    },

}

samples = atom.build_dataset(defs, skim=skim, protocol='root', cache=False)

[Back to contents](#contents)

Define function to get data from files

In [ ]:
def get_data_from_files():

    data = {} # define empty dictionary to hold awkward arrays
    for s in samples: # loop over samples
        print('Processing '+s+' samples') # print which sample
        frames = [] # define empty list to hold data
        for val in samples[s]['list']: # loop over each file
            temp = read_file(val,s) # call the function read_file defined below
            frames.append(temp) # append array returned from read_file to list of arrays
        data[s] = ak.concatenate(frames) # dictionary entry is concatenated arrays

    return data # return dictionary of awkward arrays

[Back to contents](#contents)

Define function to calculate the weight of each MC event.

The cross-section, k-factor, filter efficiency and sum of weights of each sample are stored in the files themselves, next to the event weights and scale factors.

In [ ]:
# variables to multiply together to give the weight of each MC event
weight_variables = ["filteff","kfac","xsec","mcWeight",
                    "ScaleFactor_PILEUP","ScaleFactor_ELE",
                    "ScaleFactor_MUON","ScaleFactor_LepTRIGGER"]

def calc_weight(events):
    total_weight = lumi*1000/events["sum_of_weights"] # *1000 to go from fb-1 to pb-1
    for variable in weight_variables:
        total_weight = total_weight*events[variable]
    return total_weight

Define function to calculate 4-lepton invariant mass

In [ ]:
def calc_mllll(lep_pt,lep_eta,lep_phi,lep_e):
    # build the 4-momentum of each lepton
    p4 = vector.zip({"pt": lep_pt, "eta": lep_eta, "phi": lep_phi, "E": lep_e})
    # first lepton in each event is [:, 0], 2nd lepton is [:, 1] etc
    return (p4[:, 0] + p4[:, 1] + p4[:, 2] + p4[:, 3]).M # invariant mass in GeV

[Back to contents](#contents)

<a id='changing_cut'></a>

## Changing a cut

If you change a cut: Cell -> Run All Below

If you change a cut here, you also need to make sure the cut is applied in the "[Applying a cut](#applying_cut)" cell.

In [ ]:
# cut on the trigger
# keep events where a single-electron or single-muon trigger fired, and a lepton is matched to it
def cut_trigger(trigE,trigM,lep_isTrigMatched):
# throw away when no trigger fired, or no lepton is matched to it
    fired = (trigE != 0) | (trigM != 0)
    matched = ak.any(lep_isTrigMatched != 0, axis=1)
    return ~(fired & matched)

# cut on lepton quality
# electrons (lep_type 11) need loose identification, muons (lep_type 13) medium identification,
# and all leptons need loose isolation and to come from the primary vertex (TTVA)
def cut_lep_quality(lep_type,lep_isLooseID,lep_isMediumID,lep_isLooseIso,lep_passTTVA):
# throw away when any of the 4 leptons fails
    good = ( ((lep_type == 11) & (lep_isLooseID != 0)) | ((lep_type == 13) & (lep_isMediumID != 0)) ) \
           & (lep_isLooseIso != 0) & (lep_passTTVA != 0)
    return ak.sum(good, axis=1) != 4

# cut on lepton charge
# paper: "selecting two pairs of isolated leptons, each of which is comprised of two leptons with the same flavour and opposite charge"
def cut_lep_charge(lep_charge):
# throw away when sum of lepton charges is not equal to 0
# first lepton in each event is [:, 0], 2nd lepton is [:, 1] etc
    return lep_charge[:, 0] + lep_charge[:, 1] + lep_charge[:, 2] + lep_charge[:, 3] != 0

# cut on lepton type
# paper: "selecting two pairs of isolated leptons, each of which is comprised of two leptons with the same flavour and opposite charge"
def cut_lep_type(lep_type):
# for an electron lep_type is 11
# for a muon lep_type is 13
# throw away when none of eeee, mumumumu, eemumu
    sum_lep_type = lep_type[:, 0] + lep_type[:, 1] + lep_type[:, 2] + lep_type[:, 3]
    return (sum_lep_type != 44) & (sum_lep_type != 48) & (sum_lep_type != 52)

[Back to contents](#contents)

<a id='applying_cut'></a>

## Applying a cut
If you add a cut: Cell -> Run All Below

In [ ]:
# variables to read from the files
variables = ['lep_pt','lep_eta','lep_phi','lep_e','lep_charge','lep_type',
             'trigE','trigM','lep_isTrigMatched',
             'lep_isLooseID','lep_isMediumID','lep_isLooseIso','lep_passTTVA',
             # add more variables here if you make cuts on them
            ]

def read_file(path,sample):
    start = time.time() # start the clock
    print("\tProcessing: "+path.split('/')[-1]) # print which file is being processed
    is_data = (sample == 'data')
    tree = uproot.open(path + ":analysis") # open the tree called analysis
    numevents = tree.num_entries # number of events
    branches = variables + ([] if is_data else weight_variables + ["sum_of_weights"]) # MC also needs the weights
    sample_data = [] # define empty list to hold the arrays of all batches for this file
    for data in tree.iterate(branches, # variables to read
                             library="ak", # choose output type as awkward array
                             entry_stop=numevents*fraction): # process up to numevents*fraction

        nIn = len(data) # number of events in this batch

        # cut on the trigger using the function cut_trigger defined above
        data = data[~cut_trigger(data.trigE, data.trigM, data.lep_isTrigMatched)]

        # cut on lepton quality using the function cut_lep_quality defined above
        data = data[~cut_lep_quality(data.lep_type, data.lep_isLooseID, data.lep_isMediumID,
                                     data.lep_isLooseIso, data.lep_passTTVA)]

        # cut on lepton charge using the function cut_lep_charge defined above
        data = data[~cut_lep_charge(data.lep_charge)]

        # cut on lepton type using the function cut_lep_type defined above
        data = data[~cut_lep_type(data.lep_type)]

        # calculation of 4-lepton invariant mass using the function calc_mllll defined above
        data['mllll'] = calc_mllll(data.lep_pt, data.lep_eta, data.lep_phi, data.lep_e)

        if not is_data: # only do this for Monte Carlo simulation files
            # multiply all Monte Carlo weights and scale factors together to give total weight
            data['totalWeight'] = calc_weight(data)

        # array contents can be printed at any stage like this
        #print(data)

        # a single variable can be printed at any stage like this
        #print(data['lep_pt'])

        nOut = len(data) # number of events passing cuts in this batch
        sample_data.append(data) # append array from this batch to the list for the whole file
        elapsed = time.time() - start # time taken to process
        print("\t\t nIn: "+str(nIn)+",\t nOut: \t"+str(nOut)+"\t in "+str(round(elapsed,1))+"s") # events before and after

    return ak.concatenate(sample_data) # return array containing events passing all cuts

[Back to contents](#contents)

This is where the processing happens (this will take some minutes)

In [ ]:
start = time.time() # time at start of whole processing
data = get_data_from_files() # process all files
elapsed = time.time() - start # time after whole processing
print("Time taken: "+str(round(elapsed,1))+"s") # print total time taken to process every file

Processing data samples
	Processing: data_A


[Back to contents](#contents)

<a id='plotting'></a>

## Plotting
If you only want a make a change in plotting: Cell -> Run All Below

Define class to display log values

In [ ]:
# define class to display 1 and 10 normally
class CustomTicker(LogFormatterSciNotation): 
    def __call__(self, x, pos=None): 
        if x not in [1,10]: # not 1 or 10
            return LogFormatterSciNotation.__call__(self,x, pos=None)
        else: # 1 or 10
            return "{x:g}".format(x=x) # standard notation

Define function to plot the data

In [ ]:
def plot_data(data):

    xmin = 130 # GeV
    xmax = 1230 # GeV
    step_size = 55 # GeV

    bin_edges = np.arange(start=xmin, # The interval includes this value
                     stop=xmax+step_size, # The interval doesn't include this value
                     step=step_size ) # Spacing between values
    bin_centres = np.arange(start=xmin+step_size/2, # The interval includes this value
                            stop=xmax+step_size/2, # The interval doesn't include this value
                            step=step_size ) # Spacing between values

    data_x,_ = np.histogram(ak.to_numpy(data['data']['mllll']), 
                            bins=bin_edges ) # histogram the data
    data_x_errors = np.sqrt( data_x ) # statistical error on the data

    signal_x = ak.to_numpy(data['Graviton']['mllll']) # histogram the signal
    signal_weights = ak.to_numpy(data['Graviton'].totalWeight) # get the weights of the signal events
    signal_color = samples['Graviton']['color'] # get the colour for the signal bar

    mc_x = [] # define list to hold the Monte Carlo histogram entries
    mc_weights = [] # define list to hold the Monte Carlo weights
    mc_colors = [] # define list to hold the colors of the Monte Carlo bars
    mc_labels = [] # define list to hold the legend labels of the Monte Carlo bars

    for s in samples: # loop over samples
        if s not in ['data', 'Graviton']: # if not data nor signal
            mc_x.append( ak.to_numpy(data[s]['mllll']) ) # append to the list of Monte Carlo histogram entries
            mc_weights.append( ak.to_numpy(data[s].totalWeight) ) # append to the list of Monte Carlo weights
            mc_colors.append( samples[s]['color'] ) # append to the list of Monte Carlo bar colors
            mc_labels.append( s ) # append to the list of Monte Carlo legend labels
    


    # *************
    # Main plot 
    # *************
    main_axes = plt.gca() # get current axes
    
    # plot the data points
    main_axes.errorbar(x=bin_centres, y=data_x, yerr=data_x_errors,
                       fmt='ko', # 'k' means black and 'o' is for circles 
                       label='Data') 
    
    # plot the Monte Carlo bars
    mc_heights = main_axes.hist(mc_x, bins=bin_edges, 
                                weights=mc_weights, stacked=True, 
                                color=mc_colors, label=mc_labels )
    
    mc_x_tot = mc_heights[0][-1] # stacked background MC y-axis value
    
    # calculate MC statistical uncertainty: sqrt(sum w^2)
    mc_x_err = np.sqrt(np.histogram(np.hstack(mc_x), bins=bin_edges, weights=np.hstack(mc_weights)**2)[0])
    
    # plot the signal bar
    main_axes.hist(signal_x, bins=bin_edges, bottom=mc_x_tot, 
                   weights=signal_weights, color=signal_color,
                   label='Graviton')
    
    # plot the statistical uncertainty
    main_axes.bar(bin_centres, # x
                  2*mc_x_err, # heights
                  alpha=0.5, # half transparency
                  bottom=mc_x_tot-mc_x_err, color='none', 
                  hatch="////", width=step_size, label='Stat. Unc.' )

    # set the x-limit of the main axes
    main_axes.set_xlim( left=xmin, right=xmax ) 
    
    # separation of x axis minor ticks
    main_axes.xaxis.set_minor_locator( AutoMinorLocator() ) 
    
    # set the axis tick parameters for the main axes
    main_axes.tick_params(which='both', # ticks on both x and y axes
                          direction='in', # Put ticks inside and outside the axes
                          top=True, # draw ticks on the top axis
                          right=True ) # draw ticks on right axis
    
    # x-axis label
    main_axes.set_xlabel(r'4-lepton invariant mass $\mathrm{m_{4l}}$ [GeV]',
                        fontsize=13, x=1, horizontalalignment='right' )
    
    # write y-axis label for main axes
    main_axes.set_ylabel('Events / '+str(step_size)+' GeV',
                         y=1, horizontalalignment='right') 
    
    # add minor ticks on y-axis for main axes
    main_axes.yaxis.set_minor_locator( AutoMinorLocator() ) 
    
    main_axes.set_yscale('log') # set y-scale
    smallest_contribution = mc_heights[0][0] # get smallest contribution
    smallest_contribution.sort() # sort smallest contribution
    bottom = np.amax(data_x)/1000 # set bottom limit on y-axis
    top = np.amax(data_x)*100 # set top limit on y-axis
    main_axes.set_ylim( bottom=bottom, top=top ) # y-axis limits
    main_axes.yaxis.set_major_formatter( CustomTicker() ) 
    locmin = LogLocator(base=10.0, # log base 10
                        subs=(0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9) ) # minor tick every 0.1 
    main_axes.yaxis.set_minor_locator( locmin ) # set minor ticks

    # Add text 'ATLAS Open Data' on plot
    plt.text(0.05, # x
             0.93, # y
             'ATLAS Open Data', # text
             transform=main_axes.transAxes, # coordinate system used is that of main_axes
             fontsize=13 ) 
    
    # Add text 'for education' on plot
    plt.text(0.05, # x
             0.88, # y
             'for education', # text
             transform=main_axes.transAxes, # coordinate system used is that of main_axes
             style='italic',
             fontsize=8 ) 
    
    # Add energy and luminosity
    lumi_used = str(round(lumi*fraction,2)) # luminosity to write on the plot
    plt.text(0.05, # x
             0.82, # y
             r'$\sqrt{s}$=13 TeV, '+lumi_used+' fb$^{-1}$', # text
             transform=main_axes.transAxes ) # coordinate system used is that of main_axes
    
    # Add a label for the analysis carried out
    plt.text(0.05, # x
             0.75, # y
             r'$G \rightarrow ZZ \rightarrow l^+l^-l^+l^-$', # text 
             transform=main_axes.transAxes ) # coordinate system used is that of main_axes

    # draw the legend
    main_axes.legend(ncol=2, # 2 columns
                     frameon=False ) # no box around the legend
    
    return

[Back to contents](#contents)

Call the function to plot the data

In [ ]:
plot_data(data)

[Back to contents](#contents)

<a id='going_further'></a>

## What can you do to explore this analysis?

* Increase the fraction of data used in '[Lumi, fraction, file path](#fraction)'
* Check how many events are being thrown away by each cut in '[Applying a cut](#applying_cut)'
* Add more cuts from the [ATLAS paper searching for ZZ resonances](https://link.springer.com/article/10.1140%2Fepjc%2Fs10052-018-5686-3#Sec5) in '[Changing a cut](#changing_cut)' and '[Applying a cut](#applying_cut)'
* Add a plot to show the ratio between Data and MC other than Graviton like [Figure 4 of the ATLAS paper searching for ZZ resonances](https://link.springer.com/article/10.1140%2Fepjc%2Fs10052-018-5686-3#Fig4)
* Get the estimated numbers of events, similar to [Table 4 of the ATLAS paper searching for ZZ resonances](https://link.springer.com/article/10.1140/epjc/s10052-018-5686-3/tables/4)
* Split the analysis into ggF and VBF, like [Section 5 of the ATLAS paper searching for ZZ resonances](https://link.springer.com/article/10.1140%2Fepjc%2Fs10052-018-5686-3#Sec5)
* Your idea!

[Back to contents](#contents)